# Phân tích Thị trường Chứng khoán sử dụng PySpark

## 1. Giới thiệu

### Bài toán phân tích chứng khoán
Notebook này phân tích **thị trường chứng khoán đa mã (multi-stock)** và xây dựng mô hình dự đoán trên dữ liệu time series đã được xử lý sẵn. 
Mục tiêu chính của dự án này là:

- **Khám phá xu hướng và mức độ biến động** của nhiều mã cổ phiếu theo thời gian
- **So sánh các ticker** để tìm mối liên hệ giữa thị trường Việt Nam và Mỹ
- **Tính toán các chỉ báo kỹ thuật bổ sung** từ dữ liệu features đã có sẵn
- **Xây dựng mô hình dự đoán** giá và xu hướng tăng/giảm của cổ phiếu
- **Đánh giá hiệu suất mô hình** bằng các chỉ số như AUC, Accuracy, Precision, và Recall

### Tại sao sử dụng PySpark?

**PySpark (Apache Spark)** là framework xử lý dữ liệu phân tán mạnh mẽ. Những lợi thế của PySpark:

1. **Xử lý dữ liệu lớn**: Spark có thể xử lý hàng tỷ dòng dữ liệu một cách hiệu quả
2. **Tốc độ cao**: Spark sử dụng in-memory caching, tốc độ nhanh hơn Hadoop MapReduce 10-100 lần
3. **Tính năng SQL**: Spark SQL cho phép viết code structured và tối ưu
4. **Window Functions**: Hỗ trợ các hàm cửa sổ để tính toán chỉ báo theo từng ticker
5. **Machine Learning**: MLlib cung cấp các thuật toán ML scalable
6. **Streaming**: Spark Streaming hỗ trợ xử lý dữ liệu real-time

### Tại sao sử dụng định dạng Parquet?

**Parquet** là định dạng lưu trữ cột (Columnar Storage Format). Ưu điểm:

1. **Nén dữ liệu tốt**: Giảm kích thước file lên tới 10x
2. **Truy vấn nhanh**: Chỉ cần đọc các cột cần thiết, không phải toàn bộ dòng
3. **Schema preservation**: Lưu giữ kiểu dữ liệu chính xác
4. **Tương thích cao**: Hoạt động tốt với Spark, Hive, Presto, v.v.
5. **Phù hợp cho pipeline features**: Lưu trực tiếp dữ liệu đã feature engineering

---

## 2. Khởi tạo Spark Session

SparkSession là điểm vào chính để làm việc với Spark. Chúng tôi sẽ cấu hình:
- **appName**: Tên ứng dụng
- **master**: Chế độ chạy (local cho máy tính cá nhân)
- **spark.sql.adaptive.enabled**: Tối ưu hóa truy vấn SQL tự động

In [1]:
# Nhập các thư viện cần thiết
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import (
    col, when, lag, lead, sum as spark_sum, avg, stddev,
    round, lit
 )
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình matplotlib để hiển thị biểu đồ
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("StockMarketAnalysisPySpark") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.shuffle.partitions", "200") \
    .getOrCreate()

# Thiết lập log level (giảm độ chi tiết của log)
spark.sparkContext.setLogLevel("WARN")

print("✓ SparkSession đã được khởi tạo thành công!")
print(f"✓ Spark Version: {spark.version}")
print(f"✓ Python Version: {spark.sparkContext.pythonExec}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 02:13:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✓ SparkSession đã được khởi tạo thành công!
✓ Spark Version: 4.1.1
✓ Python Version: python3


## 3. Đọc dữ liệu từ Parquet

Notebook sẽ sử dụng trực tiếp file Parquet features đã được tạo sẵn: `data_stocks_features.parquet`.
Đây là dataset đa mã (multi-stock) đã qua tiền xử lý, phù hợp để phân tích và huấn luyện mô hình time series.

### Cấu trúc dữ liệu kỳ vọng
Dataset bao gồm các cột sau:
- `time`: thời gian giao dịch (timestamp)
- `open`, `high`, `low`, `close`: giá OHLC
- `volume`: khối lượng giao dịch
- `ticker`: mã cổ phiếu
- `prev_close`: giá đóng cửa phiên trước
- `ma7`: moving average 7 phiên
- `daily_return`: tỷ suất sinh lợi hàng ngày

Vì đây là dữ liệu multi-stock, các bước xử lý tiếp theo sẽ luôn được thực hiện theo từng `ticker` và theo thứ tự thời gian để tránh leakage.

In [ ]:
# Đọc cố định từ file features như yêu cầu
parquet_path = "data_stocks_features.parquet"
df = spark.read.parquet(parquet_path)

# Chuẩn hóa cột thời gian để nhất quán với toàn bộ notebook
if "time" not in df.columns and "date" in df.columns:
    df = df.withColumnRenamed("date", "time")
if "time" in df.columns:
    df = df.withColumn("time", col("time").cast("timestamp"))

print(f"Đã load dữ liệu từ: {parquet_path}")
print("═" * 80)
print("THÔNG TIN SCHEMA - CẤU TRÚC DỮ LIỆU")
print("═" * 80)
df.printSchema()

print("\n" + "═" * 80)
print("THÔNG TIN TỔNG QUÁT DATAFRAME")
print("═" * 80)
print(f"Tổng số dòng: {df.count():,}")
print(f"Tổng số cột: {len(df.columns)}")
print(f"Tên các cột: {df.columns}")

print("\n" + "═" * 80)
print("DỮ LIỆU MẪU (10 DÒNG ĐẦU TIÊN)")
print("═" * 80)
df.show(10, truncate=False)

Đã load dữ liệu từ: data_stocks_features.parquet
════════════════════════════════════════════════════════════════════════════════
THÔNG TIN SCHEMA - CẤU TRÚC DỮ LIỆU
════════════════════════════════════════════════════════════════════════════════
root
 |-- time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)
 |-- prev_close: double (nullable = true)
 |-- ma7: double (nullable = true)
 |-- daily_return: double (nullable = true)


════════════════════════════════════════════════════════════════════════════════
THÔNG TIN TỔNG QUÁT DATAFRAME
════════════════════════════════════════════════════════════════════════════════


### 3.1 Sắp xếp và kiểm tra dữ liệu

Dù file Parquet ban đầu chưa sắp xếp theo time series, ta sẽ chủ động sắp xếp lại để tính đúng các chỉ báo kỹ thuật.
- Nếu có `ticker`: sắp xếp theo (`ticker`, `time`).
- Nếu không có `ticker`: sắp xếp theo `time`.

In [ ]:
# Sắp xếp dữ liệu để đảm bảo đúng thứ tự time series
if "ticker" in df.columns:
    df_sorted = df.orderBy(col("ticker"), col("time"))
    print("✓ Dữ liệu đã được sắp xếp theo (ticker, time)")
    df_sorted.select("ticker", "time", "open", "high", "low", "close", "volume").limit(5).show()
else:
    df_sorted = df.orderBy(col("time"))
    print("✓ Dữ liệu đã được sắp xếp theo time")
    df_sorted.select("time", "open", "high", "low", "close", "volume").limit(5).show()

print("\nLưu ý: Không shuffle dữ liệu time series để tránh data leakage.")

## 4. Khám phá dữ liệu (EDA)

Trong phần này, chúng ta sẽ:
1. Tính toán các thống kê mô tả (mean, stddev, min, max, v.v.)
2. Kiểm tra giá trị thiếu (null values)
3. Phân tích phân phối dữ liệu

In [ ]:
print("═" * 80)
print("THỐNG KÊ MÔ TẢ (DESCRIPTIVE STATISTICS)")
print("═" * 80)

# Sử dụng describe() để lấy thống kê cơ bản
df_sorted.select(["open", "high", "low", "close", "volume", "prev_close", "ma7", "daily_return"]).describe().show()

print("\n" + "═" * 80)
print("KIỂM TRA GIÁ TRỊ THIẾU (NULL VALUES)")
print("═" * 80)

# Kiểm tra giá trị null cho mỗi cột
null_counts = df_sorted.select(
    [spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sorted.columns]
).collect()[0]

for col_name in df_sorted.columns:
    null_count = null_counts[col_name]
    null_percentage = (null_count / df_sorted.count()) * 100
    print(f"{col_name:15} | Null count: {null_count:6,} | Null %: {null_percentage:6.2f}%")

print("\n✓ Không có giá trị null trong dữ liệu" if null_counts[0] == 0 else "⚠ Có các giá trị null cần xử lý")

In [ ]:
# Tính toán các chỉ số bổ sung
from pyspark.sql.functions import min as spark_min, max as spark_max, count, countDistinct

print("═" * 80)
print("PHÂN TÍCH CHI TIẾT HƠN")
print("═" * 80)

stats = df_sorted.agg(
    count("time").alias("Total Records"),
    countDistinct("time").alias("Unique Timestamps"),
    spark_min("time").alias("Start Time"),
    spark_max("time").alias("End Time"),
    round(avg("volume"), 2).alias("Avg Volume"),
    spark_min("volume").alias("Min Volume"),
    spark_max("volume").alias("Max Volume"),
    round(avg("close"), 4).alias("Avg Close Price"),
    spark_min("close").alias("Min Price"),
    spark_max("close").alias("Max Price")
).collect()[0]

for key, value in stats.asDict().items():
    print(f"{key:20} : {value}")

## 5. Feature Engineering - Tính toán các chỉ số kỹ thuật

Dataset đã có sẵn `prev_close`, `ma7` và `daily_return`, nên phần này sẽ bổ sung các đặc trưng time series nâng cao theo từng ticker:
1. Moving Average dài hơn: MA14, MA30
2. Volatility cuộn (rolling std)
3. Momentum
4. Lag features
5. RSI và MACD để tăng sức mạnh mô hình

Tất cả phép tính đều dùng Window partition theo `ticker` và order theo `time`.

In [ ]:
# Định nghĩa Window theo từng ticker và thời gian
window_base = Window.partitionBy("ticker").orderBy("time")
window_14 = window_base.rowsBetween(-13, 0)
window_30 = window_base.rowsBetween(-29, 0)

# Tính toán các feature bổ sung trên dữ liệu đã có sẵn feature gốc
df_features = (
    df_sorted
    .withColumn("ma14", round(avg("close").over(window_14), 4))
    .withColumn("ma30", round(avg("close").over(window_30), 4))
    .withColumn("rolling_volatility_14", round(stddev("daily_return").over(window_14), 6))
    .withColumn("momentum_3", round(col("close") - lag("close", 3).over(window_base), 4))
    .withColumn("lag_close_1", lag("close", 1).over(window_base))
    .withColumn("lag_close_3", lag("close", 3).over(window_base))
    .withColumn("lag_return_1", lag("daily_return", 1).over(window_base))
    .withColumn("lag_return_3", lag("daily_return", 3).over(window_base))
)

print("✓ Đã tính toán các feature time series bổ sung")
print("\nDữ liệu sau Feature Engineering:")
df_features.select("ticker", "time", "close", "ma7", "ma14", "ma30", "rolling_volatility_14", "momentum_3").show(20, truncate=False)

In [ ]:
# Tính RSI (Relative Strength Index) theo từng ticker
# RSI = 100 - (100 / (1 + RS))
# RS = Average Gain / Average Loss (trong 14 ngày)

window_rsi = Window.partitionBy("ticker").orderBy("time").rowsBetween(-13, 0)  # 14 phiên

df_rsi = (
    df_features
    .withColumn("price_change", col("close") - lag("close", 1).over(Window.partitionBy("ticker").orderBy("time")))
    .withColumn("gain", when(col("price_change") > 0, col("price_change")).otherwise(0))
    .withColumn("loss", when(col("price_change") < 0, -col("price_change")).otherwise(0))
    .withColumn("avg_gain", round(avg("gain").over(window_rsi), 6))
    .withColumn("avg_loss", round(avg("loss").over(window_rsi), 6))
    .withColumn(
        "rsi",
        round(
            when(col("avg_loss") == 0, lit(100)).otherwise(
                100 - (100 / (1 + (col("avg_gain") / col("avg_loss"))))
            ),
            2
        )
    )
)

print("✓ Đã tính toán RSI (Relative Strength Index)")
print("\nDữ liệu với RSI:")
df_rsi.select("ticker", "time", "close", "avg_gain", "avg_loss", "rsi").show(15, truncate=False)

In [ ]:
# Tính MACD xấp xỉ bằng rolling mean để phù hợp hoàn toàn với Spark API
# MACD = MA12 - MA26
# Signal = MA9 của MACD
# Histogram = MACD - Signal

window_12 = window_base.rowsBetween(-11, 0)
window_26 = window_base.rowsBetween(-25, 0)
window_9 = window_base.rowsBetween(-8, 0)

df_macd = (
    df_rsi
    .withColumn("ma12", round(avg("close").over(window_12), 6))
    .withColumn("ma26", round(avg("close").over(window_26), 6))
    .withColumn("macd", round(col("ma12") - col("ma26"), 6))
    .withColumn("macd_signal", round(avg("macd").over(window_9), 6))
    .withColumn("macd_histogram", round(col("macd") - col("macd_signal"), 6))
)

print("✓ Đã tính toán MACD (rolling mean version)")
print("\nDữ liệu với MACD:")
df_macd.select("ticker", "time", "close", "ma12", "ma26", "macd", "macd_signal", "macd_histogram").show(15, truncate=False)

## 6. Tạo nhãn (Label) cho mô hình phân loại

Chúng tôi sẽ tạo nhãn nhị phân theo từng `ticker`:
- **Label = 1**: Nếu giá đóng cửa phiên kế tiếp > phiên hiện tại (giá tăng)
- **Label = 0**: Nếu giá đóng cửa phiên kế tiếp ≤ phiên hiện tại (giá không tăng)

In [ ]:
# Window để lấy giá ngày tiếp theo theo từng ticker
window_next_day = Window.partitionBy("ticker").orderBy("time")

# Tạo nhãn dự đoán
df_labeled = (
    df_macd
    .withColumn("next_close", lead("close", 1).over(window_next_day))
    .withColumn("label", when(col("next_close") > col("close"), 1).otherwise(0))
)

print("✓ Đã tạo Label cho mô hình")
print("\nDữ liệu với Label:")
df_labeled.select("ticker", "time", "close", "next_close", "label").show(15, truncate=False)

print("\n" + "="*80)
print("PHÂN PHỐI NHÃN")
print("="*80)
df_labeled.groupBy("label").count().show()

# Tính phần trăm
label_counts = df_labeled.groupBy("label").count().collect()
total = df_labeled.count()

for row in label_counts:
    label = row[0]
    count = row[1]
    percentage = (count / total) * 100
    label_text = "Giá tăng" if label == 1 else "Giá không tăng"
    print(f"{label_text:20} (Label={label}): {count:,} ({percentage:.2f}%)")

## 7. Chuẩn bị Feature cho Machine Learning

Chúng tôi sẽ:
1. Chọn các feature phù hợp cho time series đa mã
2. Loại bỏ các dòng có giá trị null do rolling/lag/lead
3. Sử dụng **VectorAssembler** để gộp tất cả các features thành một cột vector

Feature chính sẽ lấy từ dữ liệu gốc và feature engineering bổ sung, toàn bộ theo từng `ticker`.

In [ ]:
# Chọn các features sẽ sử dụng cho mô hình
feature_columns = [
    "prev_close", "ma7", "daily_return",
    "ma14", "ma30", "rolling_volatility_14",
    "momentum_3", "lag_close_1", "lag_close_3",
    "lag_return_1", "lag_return_3",
    "rsi", "macd", "macd_signal", "macd_histogram",
    "volume"
]

print("Các feature sẽ sử dụng:")
for i, feat in enumerate(feature_columns, 1):
    print(f"{i:2}. {feat}")

# Loại bỏ các dòng có giá trị null sau lag/rolling/lead
df_clean = df_labeled.dropna(subset=feature_columns + ["label", "next_close"])

print(f"\n✓ Loại bỏ null values")
print(f"  - Dòng ban đầu: {df_labeled.count():,}")
print(f"  - Dòng sau khi loại bỏ null: {df_clean.count():,}")
print(f"  - Dòng bị loại bỏ: {df_labeled.count() - df_clean.count():,}")

# Sử dụng VectorAssembler để gộp các features
vector_assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_vectorized = vector_assembler.transform(df_clean)

print(f"\n✓ Đã sử dụng VectorAssembler")
print("\nDữ liệu sau VectorAssembler:")
df_vectorized.select("ticker", "time", "close", "features", "label").show(5, truncate=True)

In [ ]:
# Chuẩn hóa các features để mô hình học tốt hơn

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_vectorized)
df_scaled = scaler_model.transform(df_vectorized)

print("✓ Đã chuẩn hóa (Standardize) các features")
print("\nDữ liệu sau khi chuẩn hóa:")
df_scaled.select("ticker", "time", "close", "scaled_features", "label").show(5, truncate=True)

## 8. Chia dữ liệu và Huấn luyện nhiều mô hình

Chúng tôi sẽ:
1. Chia dữ liệu theo **thời gian** (không random shuffle)
2. Huấn luyện mô hình **Logistic Regression**
3. Huấn luyện thêm mô hình **Linear Regression** và **Random Forest** để so sánh
4. Thực hiện dự đoán trên tập test

**Lưu ý:** Với time series, không được dùng `randomSplit()` vì sẽ gây leakage.

In [ ]:
# Chia dữ liệu theo thời gian, không shuffle
# Lấy ngưỡng thời gian tương ứng 80% dữ liệu cũ nhất cho train
split_ts = df_scaled.selectExpr("percentile_approx(unix_timestamp(time), 0.8) as ts").collect()[0]["ts"]

df_train = df_scaled.filter(F.unix_timestamp("time") < F.lit(split_ts))
df_test = df_scaled.filter(F.unix_timestamp("time") >= F.lit(split_ts))

print("="*80)
print("CHIA TRAIN/TEST SET THEO THỜI GIAN")
print("="*80)
print(f"Training set: {df_train.count():,} dòng")
print(f"Test set:     {df_test.count():,} dòng")
print(f"Total:        {df_scaled.count():,} dòng")

# Kiểm tra phân phối label
print("\n" + "="*80)
print("PHÂN PHỐI LABEL TRONG TRAIN SET")
print("="*80)
df_train.groupBy("label").count().show()

print("\n" + "="*80)
print("PHÂN PHỐI LABEL TRONG TEST SET")
print("="*80)
df_test.groupBy("label").count().show()

In [ ]:
# Huấn luyện và so sánh 3 mô hình
print("\n" + "="*80)
print("HUẤN LUYỆN VÀ SO SÁNH NHIỀU MÔ HÌNH")
print("="*80)

lr_model = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.5,
    threshold=0.5
)

linear_model = LinearRegression(
    featuresCol="scaled_features",
    labelCol="next_close",
    predictionCol="predicted_next_close",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.5,
    standardization=True
)

rf_model = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)

print("\nHuấn luyện đang diễn ra...")
lr_fitted = lr_model.fit(df_train)
linear_fitted = linear_model.fit(df_train)
rf_fitted = rf_model.fit(df_train)

print("✓ Đã huấn luyện thành công 3 mô hình!")
print(f"\n[Logistic Regression] Coefficients:")
print(lr_fitted.coefficients)
print(f"Intercept: {lr_fitted.intercept}")

print("\n[Linear Regression] Coefficients:")
print(linear_fitted.coefficients)
print(f"Intercept: {linear_fitted.intercept}")

print("\n[Random Forest] Feature importances:")
print(rf_fitted.featureImportances)

In [ ]:
# Thực hiện dự đoán trên tập test cho cả 3 mô hình
df_predictions_lr = lr_fitted.transform(df_test)
df_predictions_linear_raw = linear_fitted.transform(df_test)
df_predictions_linear = (
    df_predictions_linear_raw
    .withColumn("rawPrediction", col("predicted_next_close") - col("close"))
    .withColumn("prediction", when(col("predicted_next_close") > col("close"), 1.0).otherwise(0.0))
    .withColumn("probability", when(col("prediction") == 1.0, lit(1.0)).otherwise(lit(0.0)))
)
df_predictions_rf = rf_fitted.transform(df_test)

# Giữ Logistic Regression làm kết quả mặc định cho các biểu đồ phía sau
df_predictions = df_predictions_lr

print("✓ Đã thực hiện dự đoán trên tập test cho cả 3 mô hình")
print("\nKết quả dự đoán Logistic Regression (10 dòng đầu):")
df_predictions.select(
    "ticker", "time", "close", "label", "prediction", "probability"
).show(10, truncate=False)

# Cache predictions cho việc tính toán metrics
df_predictions.cache()
df_predictions_linear.cache()
df_predictions_rf.cache()

## 9. Đánh giá hiệu suất mô hình

Chúng tôi sẽ tính toán các chỉ số đánh giá quan trọng:
- **AUC (Area Under the ROC Curve)**: Đo lường khả năng phân biệt của mô hình
- **Accuracy**: Tỷ lệ dự đoán đúng
- **Precision**: Tỷ lệ dự đoán dương tính đúng
- **Recall (Sensitivity)**: Tỷ lệ xác định đúng các trường hợp dương tính
- **F1-Score**: Trung bình hài hòa của Precision và Recall

In [ ]:
# AUC Evaluation
binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

from pyspark.sql.functions import col

def evaluate_binary_predictions(predictions_df):
    auc_value = binary_evaluator.evaluate(predictions_df)
    total_predictions = predictions_df.count()
    correct_predictions = predictions_df.filter(col("prediction") == col("label")).count()
    true_positives = predictions_df.filter((col("prediction") == 1) & (col("label") == 1)).count()
    false_positives = predictions_df.filter((col("prediction") == 1) & (col("label") == 0)).count()
    false_negatives = predictions_df.filter((col("prediction") == 0) & (col("label") == 1)).count()
    true_negatives = predictions_df.filter((col("prediction") == 0) & (col("label") == 0)).count()

    accuracy_value = correct_predictions / total_predictions if total_predictions > 0 else 0
    precision_value = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall_value = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_value = 2 * (precision_value * recall_value) / (precision_value + recall_value) if (precision_value + recall_value) > 0 else 0
    specificity_value = true_negatives / (true_negatives + false_positives) if (true_negatives + false_positives) > 0 else 0

    return {
        'AUC': auc_value,
        'Accuracy': accuracy_value,
        'Precision': precision_value,
        'Recall': recall_value,
        'F1': f1_value,
        'Specificity': specificity_value,
        'TP': true_positives,
        'FP': false_positives,
        'TN': true_negatives,
        'FN': false_negatives,
    }

metrics_lr = evaluate_binary_predictions(df_predictions_lr)
metrics_linear = evaluate_binary_predictions(df_predictions_linear)
metrics_rf = evaluate_binary_predictions(df_predictions_rf)

comparison_rows = [
    ('Logistic Regression', metrics_lr),
    ('Linear Regression', metrics_linear),
    ('Random Forest', metrics_rf),
]

comparison_data = [
    {
        'Model': model_name,
        'AUC': float(f"{metrics['AUC']:.4f}"),
        'Accuracy': float(f"{metrics['Accuracy']:.4f}"),
        'Precision': float(f"{metrics['Precision']:.4f}"),
        'Recall': float(f"{metrics['Recall']:.4f}"),
        'F1': float(f"{metrics['F1']:.4f}"),
        'Specificity': float(f"{metrics['Specificity']:.4f}"),
    } for model_name, metrics in comparison_rows
]

comparison_df = pd.DataFrame(comparison_data)

print("="*80)
print("ĐÁNH GIÁ MÔ HÌNH - SO SÁNH")
print("="*80)
print(comparison_df.to_string(index=False))

# Giữ lại biến của Logistic Regression cho các cell phía sau
auc_score = metrics_lr['AUC']
accuracy = metrics_lr['Accuracy']
precision = metrics_lr['Precision']
recall = metrics_lr['Recall']
f1_score = metrics_lr['F1']
true_positives = metrics_lr['TP']
false_positives = metrics_lr['FP']
true_negatives = metrics_lr['TN']
false_negatives = metrics_lr['FN']

print("\nChi tiết Logistic Regression:")
print(f"AUC Score: {auc_score:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"Specificity: {true_negatives / (true_negatives + false_positives):.4f}")
print(f"F1-Score: {f1_score:.4f}")
print("Chi tiet Linear Regression:")
print(f"AUC Score: {metrics_linear['AUC']:.4f}")
print(f"Accuracy: {metrics_linear['Accuracy']:.4f}")
print(f"Precision: {metrics_linear['Precision']:.4f}")
print(f"Recall (Sensitivity): {metrics_linear['Recall']:.4f}")
print(f"Specificity: {metrics_linear['Specificity']:.4f}")
print(f"F1-Score: {metrics_linear['F1']:.4f}")
print("Chi tiet Random Forest:")
print(f"AUC Score: {metrics_rf['AUC']:.4f}")
print(f"Accuracy: {metrics_rf['Accuracy']:.4f}")
print(f"Precision: {metrics_rf['Precision']:.4f}")
print(f"Recall (Sensitivity): {metrics_rf['Recall']:.4f}")
print(f"Specificity: {metrics_rf['Specificity']:.4f}")
print(f"F1-Score: {metrics_rf['F1']:.4f}")  

## 10. Trực quan hóa kết quả

Chúng tôi sẽ:
1. Chuyển đổi Spark DataFrame sang Pandas
2. Vẽ biểu đồ giá cổ phiếu và các đường MA
3. Hiển thị dự đoán và xác suất cho một ticker đại diện

In [ ]:
# Chuyển đổi dữ liệu từ Spark sang Pandas để visualize
print("Đang chuyển đổi dữ liệu từ Spark sang Pandas...")

# Lấy dữ liệu từ toàn bộ tập features đã xử lý, ưu tiên một ticker để minh họa
example_ticker = "FPT"
if df_labeled.filter(col("ticker") == example_ticker).count() == 0:
    example_ticker = df_labeled.select("ticker").first()[0]

df_full_features = (
    df_labeled.filter(col("ticker") == example_ticker)
    .select("ticker", "time", "close", "ma7", "ma14", "ma30", "rsi", "high", "low", "volume")
    .orderBy("time")
)

pandas_df = df_full_features.toPandas()
pandas_df['time'] = pd.to_datetime(pandas_df['time'])
pandas_df = pandas_df.set_index('time')

print(f"✓ Đã chuyển đổi {len(pandas_df)} dòng dữ liệu cho ticker {example_ticker}")
print("\nDữ liệu Pandas (5 dòng):")
print(pandas_df.head())

In [ ]:
# Vẽ biểu đồ giá đóng cửa và Moving Averages
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Stock Market Analysis - Technical Indicators ({example_ticker})', fontsize=16, fontweight='bold')

# Plot 1: Close Price với MA7, MA14 và MA30
ax1 = axes[0, 0]
pandas_df['close'].plot(ax=ax1, label='Close Price', color='black', linewidth=2)
pandas_df['ma7'].plot(ax=ax1, label='MA7', color='blue', linewidth=1.5)
pandas_df['ma14'].plot(ax=ax1, label='MA14', color='red', linewidth=1.5)
pandas_df['ma30'].plot(ax=ax1, label='MA30', color='green', linewidth=1.5)
ax1.set_title('Stock Price with Moving Averages', fontsize=12, fontweight='bold')
ax1.set_ylabel('Price', fontsize=10)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: RSI
ax2 = axes[0, 1]
ax2.plot(pandas_df.index, pandas_df['rsi'], label='RSI(14)', color='purple', linewidth=1.5)
ax2.axhline(y=70, color='red', linestyle='--', label='Overbought (70)', alpha=0.7)
ax2.axhline(y=30, color='green', linestyle='--', label='Oversold (30)', alpha=0.7)
ax2.set_title('Relative Strength Index (RSI)', fontsize=12, fontweight='bold')
ax2.set_ylabel('RSI Value', fontsize=10)
ax2.set_ylim([0, 100])
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

# Plot 3: Volume
ax3 = axes[1, 0]
ax3.bar(pandas_df.index, pandas_df['volume'], color='steelblue', alpha=0.7)
ax3.set_title('Trading Volume', fontsize=12, fontweight='bold')
ax3.set_ylabel('Volume', fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Price Range (High-Low)
ax4 = axes[1, 1]
price_range = pandas_df['high'] - pandas_df['low']
ax4.fill_between(pandas_df.index, 0, price_range, alpha=0.5, color='orange')
ax4.plot(pandas_df.index, price_range, color='darkorange', linewidth=1)
ax4.set_title('Daily Price Range (High - Low)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Price Range', fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('stock_analysis_technical_indicators.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Biểu đồ đã được lưu thành 'stock_analysis_technical_indicators.png'")

In [ ]:
# Vẽ biểu đồ so sánh Actual vs Predicted
# Lấy dữ liệu dự đoán từ test set
pandas_predictions = df_predictions.select(
    "ticker", "time", "close", "label", "prediction", "probability"
).orderBy("time").toPandas()

pandas_predictions['time'] = pd.to_datetime(pandas_predictions['time'])

# Tạo biểu đồ
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle(f'Model Predictions vs Actual Values ({example_ticker})', fontsize=16, fontweight='bold')

# Plot 1: Closing Price
ax1 = axes[0]
ax1.plot(pandas_predictions['time'], pandas_predictions['close'], 
         label='Actual Close Price', color='black', linewidth=2, marker='o', markersize=3)
ax1.set_title('Closing Price Over Time (Test Set)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Price', fontsize=10)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction Distribution
ax2 = axes[1]
colors = ['green' if pred == actual else 'red' 
          for pred, actual in zip(pandas_predictions['prediction'], pandas_predictions['label'])]
ax2.scatter(pandas_predictions['time'], pandas_predictions['prediction'], 
           c=colors, alpha=0.6, s=50, label='Prediction (Green=Correct, Red=Wrong)')
ax2.set_title('Model Predictions (0=Price will not increase, 1=Price will increase)', 
             fontsize=12, fontweight='bold')
ax2.set_ylabel('Prediction', fontsize=10)
ax2.set_ylim([-0.5, 1.5])
ax2.set_yticks([0, 1])
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stock_analysis_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Biểu đồ dự đoán đã được lưu thành 'stock_analysis_predictions.png'")

In [ ]:
# Tạo biểu đồ confusion matrix bằng các đếm từ Spark (không cần sklearn)

cm = [
    [true_negatives, false_positives],
    [false_negatives, true_positives]
]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Increase (0)', 'Increase (1)'],
            yticklabels=['No Increase (0)', 'Increase (1)'],
            cbar_kws={'label': 'Count'})
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix - Logistic Regression Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Confusion Matrix đã được lưu thành 'confusion_matrix.png'")
print("\n" + "="*80)
print("CONFUSION MATRIX COUNTS")
print("="*80)
print(f"True Negatives : {true_negatives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"True Positives : {true_positives}")

## 11. (Nâng cao) Xử lý dữ liệu Streaming với Spark

Phần này giới thiệu cách sử dụng **Spark Structured Streaming** để xử lý dữ liệu cổ phiếu theo thời gian thực.
Ở đây, chúng ta sẽ:
1. Đọc dữ liệu từ Parquet theo phương pháp streaming (mô phỏng dữ liệu đến từng lô)
2. Áp dụng các phép biến đổi giống như xử lý batch
3. Ghi kết quả vào hệ thống file hoặc bộ nhớ đệm

In [ ]:
# Ví dụ: Streaming từ thư mục chứa file Parquet
# Lưu ý: Phần này là minh họa. Trong thực tế, bạn sẽ đọc từ các nguồn streaming như Kafka, Socket, v.v.

print("="*80)
print("STRUCTURED STREAMING EXAMPLE")
print("="*80)

print("""
Cách sử dụng Spark Structured Streaming:

1. Đọc dữ liệu từ nguồn streaming (Kafka, Socket, File Directory, v.v.)
2. Áp dụng các phép biến đổi (transformations)
3. Ghi kết quả đến sink (Console, HDFS, Kafka, v.v.)

Ví dụ code:

# Đọc từ Kafka
df_stream = spark.readStream \\
    .format("kafka") \\
    .option("kafka.bootstrap.servers", "localhost:9092") \\
    .option("subscribe", "stock-data") \\
    .load()

# Áp dụng transformations
df_processed = df_stream.select(
    from_json(col("value").cast("string"), schema).alias("data")
) \\
    .select("data.*") \\
    .withColumn("ma5", avg("close").over(window("date", "5 days")))

# Ghi vào console
query = df_processed.writeStream \\
    .format("console") \\
    .option("checkpointLocation", "/tmp/checkpoint") \\
    .start()

query.awaitTermination()
""")

print("\n✓ Xem các example code trên để hiểu cách sử dụng Structured Streaming")
print("  Để thực hành, bạn cần:")
print("  1. Một Kafka cluster hoặc nguồn streaming khác")
print("  2. Cấu hình spark-defaults.conf để hỗ trợ streaming")
print("  3. Xác định schema chính xác của dữ liệu incoming")

In [ ]:
# Ví dụ thực hành: Mô phỏng streaming bằng cách chia batch dữ liệu
print("\n" + "="*80)
print("MOCKING STREAM PROCESSING")
print("="*80)

# Chia dữ liệu thành các batch (mô phỏng data stream)
partition_count = 5
df_sorted_temp = (
    df_labeled
    .repartition(partition_count, "ticker")
    .sortWithinPartitions("time")
)

print(f"\nDữ liệu đã được chia thành {partition_count} partition")
print(f"Mỗi partition đại diện cho một batch trong stream")
print(f"\nTrong thực tế, bạn sẽ:")
print("1. Đọc dữ liệu từ Kafka streaming")
print("2. Áp dụng Window Functions tương tự")
print("3. Ghi kết quả vào sink (e.g., HDFS, Cassandra, etc.)")
print("\n✓ Điều này cho phép xử lý dữ liệu real-time mà không cần lưu toàn bộ vào bộ nhớ")

## 12. Tóm kết và Đề xuất

Notebook đã phân tích dữ liệu features đa mã, xây dựng thêm chỉ báo kỹ thuật theo từng ticker và huấn luyện mô hình dự đoán theo nguyên tắc time series.

In [ ]:
print("="*80)
print("TÓM KẾT DỰ ÁN")
print("="*80)

print("""
✓ 1. KHỞI TẠO SPARK: Tạo SparkSession với cấu hình tối ưu

✓ 2. LOAD DỮ LIỆU: Đọc dữ liệu Parquet features từ data_stocks_features.parquet
   - Tổng số bản ghi: {0:,}
   - Khoảng thời gian: từ {1} đến {2}

✓ 3. EDA (Exploratory Data Analysis): 
   - Kiểm tra không có giá trị null
   - Phân tích thống kê cơ bản (mean, stddev, min, max)
   - Khối lượng giao dịch trung bình: {3:,.0f}

✓ 4. FEATURE ENGINEERING:
   - MA14, MA30
   - Rolling Volatility
   - Momentum và Lag Features
   - RSI và MACD

✓ 5. LABEL CREATION:
   - Nhãn nhị phân theo từng ticker (0: không tăng, 1: tăng)

✓ 6. FEATURE PREPARATION:
   - Vector Assembly cho các features time series
   - Standardization (mean=0, std=1)

✓ 7. MODEL TRAINING:
   - Logistic Regression với train/test split theo thời gian

✓ 8. MODEL EVALUATION:
   - AUC Score: {4:.4f}
   - Accuracy: {5:.4f}
   - Precision: {6:.4f}
   - Recall: {7:.4f}
   - F1-Score: {8:.4f}

✓ 9. VISUALIZATION:
   - Biểu đồ giá cổ phiếu với MA7, MA14 và MA30
   - Chỉ số RSI
   - Prediction vs Actual

✓ 10. STREAMING (ADVANCED):
   - Giới thiệu Spark Structured Streaming
   - Mô phỏng stream processing
""".format(
    int(df_labeled.count()),
    df_labeled.selectExpr('min(time) as min_time').collect()[0]['min_time'] if df_labeled.count() else 'N/A',
    df_labeled.selectExpr('max(time) as max_time').collect()[0]['max_time'] if df_labeled.count() else 'N/A',
    stats['Avg Volume'],
    auc_score,
    accuracy,
    precision,
    recall,
    f1_score
))

print("\n" + "="*80)
print("ĐỀ XUẤT CẢI THIỆN")
print("="*80)
print("""
1. FEATURE ENGINEERING:
   - Thử thêm nhiều lag hơn (t-5, t-10)
   - Bổ sung ATR, Bollinger Bands, volume ratios

2. MODEL IMPROVEMENT:
   - Thử XGBoost, Random Forest, Gradient Boosting
   - Walk-forward validation thay vì chỉ chia 1 lần

3. DATA HANDLING:
   - Xử lý class imbalance nếu có
   - Theo dõi model drift theo từng ticker

4. DEPLOYMENT:
   - Lưu mô hình đã huấn luyện để tái sử dụng
   - Kết hợp với Spark Streaming cho dự báo realtime
""")

print("\n" + "="*80)
print("✓ DỰ ÁN HOÀN THÀNH THÀNH CÔNG!")
print("="*80)

In [ ]:
spark.stop()